# Phase 5 -- Entity Model Experiments

Alert Intelligence Engine -- master plan Phase 5 (section 20): "IF/AE/benchmarks + validation." Gate: champion selected.

**First model training in this project.** Everything before this phase built representations; this phase trains and compares anomaly/novelty models on the Phase 3 entity representation (Customer Name + Transaction Name alerts, combined per master plan section 2's "shared representation layer... retaining alert-type context").

**PII safety note** (see incident write-up in `notebooks/02_data_pipeline.ipynb` history): this notebook never prints a raw name, DOB, or UIN. Inspection tables below use `record_id` + non-identifying context (screening list, alert type, country-level nationality, source sheet, score) only.

**Non-negotiable rules in force** (master plan section 1):
- Isolation Forest is the *starting candidate*, not a pre-declared winner -- benchmarked against Autoencoder, LOF, and One-Class SVM under identical conditions.
- No classification accuracy, precision, recall, F1, ROC-AUC, or PR-AUC -- there is no trustworthy label (Phase 1 finding).
- Champion selection uses only the unsupervised evidence framework (master plan section 11): distribution sanity, bootstrap ranking stability, repeat-customer representation consistency, cost. `Status`/`UPS`/`Released` is used **only** in one clearly-labeled exploratory cell at the end, and never feeds selection.
- Group-split (unseen customers) and time-forward split (repeat customers) are run and reported **separately** -- never a random row-level split (repeated customers/UINs would leak).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Build the combined entity dataset

In [2]:
from pipelines.normalization.pipeline import run_phase2_pipeline
from pipelines.entity.combined_dataset import build_combined_entity_dataset

normalized_sheets, phase2_report = run_phase2_pipeline(REPO_ROOT, persist=False)
assert phase2_report["overall_status"] == "PASS"

combined = build_combined_entity_dataset(
    normalized_sheets["CustomerViolation"], normalized_sheets["TransactionNameViolation"]
)

# Attach Status positionally (same row order build_combined_entity_dataset used --
# CustomerViolation rows then TransactionNameViolation rows, each in original order)
# for the ONE exploratory-only cell in section 8. NOT a feature, never in SHARED_COLUMNS.
# NOTE: record_id is NOT a safe join key for this -- Phase 2 deliberately left the 537
# exact-duplicate TransactionNameViolation rows undeduplicated (detect, don't silently
# drop), so identical raw content hashes to the same record_id and a record_id-based
# merge fans out (confirmed: 4,397 rows but only 3,852 unique record_id values).
# Positional concat avoids the ambiguity entirely.
combined["_status_exploratory_only"] = pd.concat([
    normalized_sheets["CustomerViolation"]["Alert Status"],
    normalized_sheets["TransactionNameViolation"]["Alert Status"],
], ignore_index=True)

print(f"Combined entity dataset: {len(combined)} rows")
print(combined["alert_source_sheet"].value_counts())
print(f"record_id uniqueness: {combined['record_id'].nunique()}/{len(combined)} "
      f"(duplicates expected -- Phase 2 detected but did not drop exact-duplicate rows)")

Combined entity dataset: 4397 rows
alert_source_sheet
CustomerViolation           2397
TransactionNameViolation    2000
Name: count, dtype: int64
record_id uniqueness: 3852/4397 (duplicates expected -- Phase 2 detected but did not drop exact-duplicate rows)


## 2. Two validation scenarios (reported separately, per master plan Experiment E)

1. **Unseen customers** -- group split by `customer_id`, no customer appears in both train and test.
2. **Repeat customers** -- time-forward split, same customer can appear on both sides, but every test-side alert is strictly later in time than every train-side alert.

In [3]:
from pipelines.entity.validation_splits import group_split_by_customer, time_forward_split

group_train_idx, group_test_idx = group_split_by_customer(combined, test_size=0.25)
time_train_idx, time_test_idx = time_forward_split(
    combined, "Alert Generated Date & Time (Parsed)", test_frac=0.25
)

scenarios = {
    "unseen_customers": (group_train_idx, group_test_idx),
    "repeat_customers_time_forward": (time_train_idx, time_test_idx),
}
for name, (tr, te) in scenarios.items():
    print(f"{name}: train={len(tr)}, test={len(te)}")

unseen_customers: train=3376, test=1021
repeat_customers_time_forward: train=3297, test=1100


## 3. Fit representations per scenario (train-only, no holdout leakage)

In [4]:
from features.entity_features import fit_entity_feature_artifacts, transform_entity_features
from pipelines.entity.anomaly_models import (
    extract_tabular_matrix, extract_name_representation_matrix, fit_svd,
)

scenario_data = {}

for scenario_name, (train_idx, test_idx) in scenarios.items():
    train_df = combined.iloc[train_idx].reset_index(drop=True)
    test_df = combined.iloc[test_idx].reset_index(drop=True)

    artifacts = fit_entity_feature_artifacts(train_df, "CombinedEntity")
    train_matrix, block_names = transform_entity_features(train_df, "CombinedEntity", artifacts)
    artifacts.feature_names = block_names
    test_matrix, _ = transform_entity_features(test_df, "CombinedEntity", artifacts)

    tab_train = extract_tabular_matrix(train_matrix, artifacts)
    tab_test = extract_tabular_matrix(test_matrix, artifacts)

    name_train = extract_name_representation_matrix(train_matrix, artifacts)
    name_test = extract_name_representation_matrix(test_matrix, artifacts)
    svd = fit_svd(name_train, n_components=50)
    svd_train = svd.transform(name_train)
    svd_test = svd.transform(name_test)

    scenario_data[scenario_name] = {
        "train_df": train_df, "test_df": test_df, "artifacts": artifacts,
        "tab_train": tab_train, "tab_test": tab_test,
        "svd_train": svd_train, "svd_test": svd_test, "svd": svd,
    }
    print(f"{scenario_name}: tabular dims={tab_train.shape[1]}, SVD name-repr dims={svd_train.shape[1]}")

unseen_customers: tabular dims=186, SVD name-repr dims=50


repeat_customers_time_forward: tabular dims=190, SVD name-repr dims=50


## 4. Experiment matrix -- E1 through E5, both scenarios

For each experiment: fit the model, score the held-out test set, and separately fit a second model on a random 80% bootstrap subsample of train to measure ranking stability (Spearman correlation between the two models' test scores) -- this is the "would this ranking hold up if we'd sampled training data slightly differently" check, never a label-based metric.

In [5]:
from pipelines.entity.anomaly_models import (
    fit_isolation_forest, score_isolation_forest,
    fit_autoencoder, score_autoencoder,
    fit_lof, score_lof,
    fit_ocsvm, score_ocsvm,
    RANDOM_STATE,
)
from pipelines.entity.evaluation import (
    score_distribution_summary, ranking_stability_between_models,
    customer_history_consistency, ExperimentResult,
)

MODEL_DEFS = {
    "E1_isolation_forest_tabular": ("tabular", "isolation_forest"),
    "E2_isolation_forest_name_svd": ("name_svd", "isolation_forest"),
    "E3_autoencoder_name_svd": ("name_svd", "autoencoder"),
    "E4_lof_name_svd": ("name_svd", "lof"),
    "E5_ocsvm_name_svd": ("name_svd", "ocsvm"),
}

def fit_and_score(model_kind, X_train, X_test):
    t0 = time.time()
    if model_kind == "isolation_forest":
        model = fit_isolation_forest(X_train)
        fit_s = time.time() - t0
        t1 = time.time()
        scores = score_isolation_forest(model, X_test)
    elif model_kind == "autoencoder":
        model = fit_autoencoder(X_train, bottleneck=8, epochs=100)
        fit_s = time.time() - t0
        t1 = time.time()
        scores = score_autoencoder(model, X_test)
    elif model_kind == "lof":
        model = fit_lof(X_train)
        fit_s = time.time() - t0
        t1 = time.time()
        scores = score_lof(model, X_test)
    elif model_kind == "ocsvm":
        model = fit_ocsvm(X_train)
        fit_s = time.time() - t0
        t1 = time.time()
        scores = score_ocsvm(model, X_test)
    else:
        raise ValueError(model_kind)
    score_s = time.time() - t1
    return model, scores, fit_s, score_s

rng = np.random.default_rng(RANDOM_STATE)
results = []
fitted_models = {}

for scenario_name, data in scenario_data.items():
    for exp_id, (repr_name, model_kind) in MODEL_DEFS.items():
        X_train = data["tab_train"] if repr_name == "tabular" else data["svd_train"]
        X_test = data["tab_test"] if repr_name == "tabular" else data["svd_test"]

        model, scores, fit_s, score_s = fit_and_score(model_kind, X_train, X_test)

        # bootstrap stability: refit on an 80% random subsample of train, score same test set
        n_train = X_train.shape[0]
        boot_idx = rng.choice(n_train, size=int(n_train * 0.8), replace=False)
        X_train_boot = X_train[boot_idx] if repr_name == "tabular" else X_train[boot_idx]
        _, scores_boot, _, _ = fit_and_score(model_kind, X_train_boot, X_test)
        stability = ranking_stability_between_models(scores, scores_boot)

        dist = score_distribution_summary(scores)
        hist_consistency = customer_history_consistency(data["test_df"], scores)

        result = ExperimentResult(
            experiment_id=f"{scenario_name}::{exp_id}",
            model_name=model_kind,
            representation=repr_name,
            validation_scenario=scenario_name,
            distribution=dist,
            stability_spearman=stability,
            history_consistency=hist_consistency,
            fit_seconds=round(fit_s, 3),
            score_seconds=round(score_s, 3),
        )
        results.append(result)
        fitted_models[result.experiment_id] = (model, scores)
        print(f"{result.experiment_id}: std={dist['std']:.4f}, stability={stability:.3f}, "
              f"fit={fit_s:.2f}s, score={score_s:.3f}s")

unseen_customers::E1_isolation_forest_tabular: std=0.0191, stability=0.944, fit=0.37s, score=0.008s


unseen_customers::E2_isolation_forest_name_svd: std=0.0310, stability=0.973, fit=0.46s, score=0.009s


/home/chpl/Documents/AI-validation/AI_validator/Alert-AI/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


unseen_customers::E3_autoencoder_name_svd: std=0.4266, stability=0.999, fit=4.88s, score=0.001s


unseen_customers::E4_lof_name_svd: std=36301.8336, stability=0.927, fit=0.20s, score=0.010s
unseen_customers::E5_ocsvm_name_svd: std=12.3428, stability=1.000, fit=0.04s, score=0.013s


repeat_customers_time_forward::E1_isolation_forest_tabular: std=0.0247, stability=0.934, fit=0.45s, score=0.008s


repeat_customers_time_forward::E2_isolation_forest_name_svd: std=0.0300, stability=0.970, fit=0.46s, score=0.007s


repeat_customers_time_forward::E3_autoencoder_name_svd: std=0.4638, stability=0.999, fit=0.12s, score=0.001s
repeat_customers_time_forward::E4_lof_name_svd: std=113764896.2026, stability=0.908, fit=0.05s, score=0.011s
repeat_customers_time_forward::E5_ocsvm_name_svd: std=10.4602, stability=0.999, fit=0.04s, score=0.013s


## 5. Results table

In [6]:
results_df = pd.DataFrame([r.as_dict() for r in results])
results_df["std"] = results_df["distribution"].apply(lambda d: d["std"])
results_df["is_degenerate"] = results_df["distribution"].apply(lambda d: d["is_degenerate"])
display_cols = ["experiment_id", "model_name", "representation", "validation_scenario",
                 "std", "is_degenerate", "stability_spearman", "fit_seconds", "score_seconds"]
print(results_df[display_cols].to_string(index=False))

                                              experiment_id       model_name representation           validation_scenario          std  is_degenerate  stability_spearman  fit_seconds  score_seconds
              unseen_customers::E1_isolation_forest_tabular isolation_forest        tabular              unseen_customers 1.911625e-02          False            0.943993        0.369          0.008
             unseen_customers::E2_isolation_forest_name_svd isolation_forest       name_svd              unseen_customers 3.100282e-02          False            0.973316        0.457          0.009
                  unseen_customers::E3_autoencoder_name_svd      autoencoder       name_svd              unseen_customers 4.266275e-01          False            0.999089        4.880          0.001
                          unseen_customers::E4_lof_name_svd              lof       name_svd              unseen_customers 3.630183e+04          False            0.927190        0.199          0.010
          

## 6. Champion selection

Rubric (master plan section 11 -- never accuracy, never Status/UPS/Released): drop degenerate models, prefer highest bootstrap ranking stability, tie-break on repeat-customer score consistency, then cost. Full reasoning persisted, not just an assertion.

In [7]:
from pipelines.entity.evaluation import select_champion

champion, rubric = select_champion(results)
print(json.dumps(rubric, indent=2, default=str))
print()
print(f"CHAMPION: {champion.experiment_id}")
print(f"  model: {champion.model_name}, representation: {champion.representation}, "
      f"scenario: {champion.validation_scenario}")

{
  "dropped_degenerate_experiment_ids": [],
  "ranked_experiment_ids_best_to_worst": [
    "unseen_customers::E5_ocsvm_name_svd",
    "repeat_customers_time_forward::E5_ocsvm_name_svd",
    "unseen_customers::E3_autoencoder_name_svd",
    "repeat_customers_time_forward::E3_autoencoder_name_svd",
    "unseen_customers::E2_isolation_forest_name_svd",
    "repeat_customers_time_forward::E2_isolation_forest_name_svd",
    "unseen_customers::E1_isolation_forest_tabular",
    "repeat_customers_time_forward::E1_isolation_forest_tabular",
    "unseen_customers::E4_lof_name_svd",
    "repeat_customers_time_forward::E4_lof_name_svd"
  ],
  "champion_experiment_id": "unseen_customers::E5_ocsvm_name_svd",
  "champion_stability_spearman": 0.9998576254910875,
  "champion_consistency_ratio": 0.24447452345438264,
  "selection_criteria_order": [
    "not degenerate",
    "highest bootstrap ranking stability (Spearman)",
    "lowest within-customer/overall score-std ratio",
    "lowest fit+score time"


## 7. Manual inspection of champion scores -- non-PII columns only

Top/bottom novelty rows shown by `record_id` and context only -- no names, no DOB, no UIN (this notebook is public).

In [8]:
from pipelines.entity.evaluation import top_n_novelty_table, novelty_by_segment, score_stability_across_time

champion_scenario = champion.validation_scenario
champion_model_kind = champion.model_name
champion_repr = champion.representation
_, champion_scores = fitted_models[champion.experiment_id]
champion_test_df = scenario_data[champion_scenario]["test_df"]

SAFE_DISPLAY_COLS = [
    "record_id", "alert_source_sheet", "Sanctions Screening List Name",
    "Alert Type", "Alerted Party Nationality (Normalized)", "Matched Screening % (Parsed)",
]

top_bottom = top_n_novelty_table(champion_test_df, champion_scores, SAFE_DISPLAY_COLS, n=10)
print(top_bottom.to_string(index=False))

       record_id       alert_source_sheet Sanctions Screening List Name            Alert Type Alerted Party Nationality (Normalized)  Matched Screening % (Parsed)  novelty_score  novelty_rank  group
a44e73d169db87c1 TransactionNameViolation                          OFAC Transaction Screening                    PALESTINE, STATE OF                        100.00     -18.000342           1.0    top
de041152ffc47821 TransactionNameViolation                            WC Transaction Screening                    PALESTINE, STATE OF                         95.00     -18.290497           3.0    top
65ad4950c34702a9 TransactionNameViolation                          OFAC Transaction Screening                    PALESTINE, STATE OF                         95.00     -18.290497           2.0    top
fc4f2a5e6e739f09        CustomerViolation                          OFAC            OnBoarding                                  EGYPT                         80.00     -19.120676           4.0    top
6c1dd

In [9]:
print("=== Novelty by screening list ===")
print(novelty_by_segment(champion_test_df, champion_scores, "Sanctions Screening List Name"))
print()
print("=== Novelty by source sheet (retains alert-type context) ===")
print(novelty_by_segment(champion_test_df, champion_scores, "alert_source_sheet"))
print()
print("=== Novelty by alert type ===")
print(novelty_by_segment(champion_test_df, champion_scores, "Alert Type"))

=== Novelty by screening list ===
                                    mean        std  count
Sanctions Screening List Name                             
STR-ISTR LIST                 -30.838993  11.556688     34
Consoldiated-Non-SDN          -37.333701  12.993736     28
PROHIBITION LIST              -37.520460  11.050427     68
CB-IEMS1                      -38.614428  10.738903     75
GOAML                         -40.598900        NaN      1
Security Council Committee    -42.156528  10.454044     59
LOCAL TERRORIST LIST          -44.325836  10.325263     16
WC                            -44.486489  11.864093    407
CB-IEMS                       -44.992268  13.534816     49
OFAC                          -46.095046  12.335657    281
INTERNAL WATCHLIST            -47.101879   8.047866      3

=== Novelty by source sheet (retains alert-type context) ===
                               mean        std  count
alert_source_sheet                                   
CustomerViolation        -42.

In [10]:
print("=== Score stability across time windows ===")
print(score_stability_across_time(
    champion_test_df, champion_scores, "Alert Generated Date & Time (Parsed)", n_bins=4
))

=== Score stability across time windows ===
             mean        std  count
period                             
0      -43.247166  13.020534    256
1      -44.131030  12.295128    255
2      -44.933972  11.766029    255
3      -40.785002  11.951512    255


## 8. Exploratory-only comparison against historical Status

**This cell does not feed champion selection and never will.** Master plan section 5: "It should not claim that Released means confirmed false positive... UPS means confirmed true match." `Status` was deliberately excluded from the combined feature dataset (`SHARED_COLUMNS`) -- it is joined back in here, after scoring, purely to sanity-check whether novelty correlates at all with historical handling. A correlation (or lack of one) here is neither required nor sufficient for the model to be considered good.

In [11]:
# Status was attached positionally to `combined` back in section 1 specifically so
# this cell never needs a record_id-based join (see that cell's note on why
# record_id isn't a safe join key -- undeduplicated exact-duplicate rows).
# champion_test_df is a positional slice of `combined`, so the column is already here.
from pipelines.entity.evaluation import exploratory_status_comparison
print(exploratory_status_comparison(champion_test_df, champion_scores, "_status_exploratory_only"))
print()
print("Reminder: UPS is not a confirmed true-match label (Phase 1 finding -- most UPS rows in")
print("this data literally say 'false positive' in the review comment). This table is context,")
print("not evidence the model 'works' or 'doesn't work'.")

                               mean        std  count
_status_exploratory_only                             
Released                 -43.262567  12.423988    980
UPS                      -43.184769  10.642626     39

Reminder: UPS is not a confirmed true-match label (Phase 1 finding -- most UPS rows in
this data literally say 'false positive' in the review comment). This table is context,
not evidence the model 'works' or 'doesn't work'.


## 9. Persist champion model

In [12]:
import joblib

champion_model, _ = fitted_models[champion.experiment_id]
out_dir = REPO_ROOT / "models" / "entity"
out_dir.mkdir(parents=True, exist_ok=True)

champion_path = out_dir / f"champion_{champion.experiment_id.replace('::', '_')}.joblib"
payload = {
    "model": champion_model,
    "model_kind": champion_model_kind,
    "representation": champion_repr,
    "svd": scenario_data[champion_scenario]["svd"] if champion_repr == "name_svd" else None,
    "entity_feature_artifacts": scenario_data[champion_scenario]["artifacts"],
    "validation_scenario": champion_scenario,
    "experiment_id": champion.experiment_id,
}
joblib.dump(payload, champion_path)
print(f"Champion persisted (gitignored, local-only): {champion_path}")

champion_manifest = {
    "experiment_id": champion.experiment_id,
    "model_kind": champion_model_kind,
    "representation": champion_repr,
    "validation_scenario": champion_scenario,
    "selection_rubric": rubric,
    "distribution": champion.distribution,
    "history_consistency": champion.history_consistency,
    "artifact_file": champion_path.name,
}
manifest_path = out_dir / "champion_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(champion_manifest, f, indent=2, default=str)
print(f"Champion manifest (gitignored): {manifest_path}")

Champion persisted (gitignored, local-only): /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/models/entity/champion_unseen_customers_E5_ocsvm_name_svd.joblib
Champion manifest (gitignored): /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/models/entity/champion_manifest.json


## 10. Phase 5 report (aggregate only -- no PII)

In [13]:
phase5_report = {
    "phase": "5_entity_experiments",
    "status": "PASS",
    "experiments_run": [r.as_dict() for r in results],
    "champion": {
        "experiment_id": champion.experiment_id,
        "model_kind": champion_model_kind,
        "representation": champion_repr,
        "validation_scenario": champion_scenario,
    },
    "selection_rubric": rubric,
    "checks": {
        "isolation_forest_not_assumed_winner": True,
        "autoencoder_benchmarked": True,
        "lof_and_ocsvm_benchmarked": True,
        "both_validation_scenarios_reported_separately": True,
        "no_classification_accuracy_reported": True,
        "status_used_only_in_labeled_exploratory_cell": True,
        "no_pii_in_this_report_or_notebook_output": True,
    },
    "known_limitations": [
        "Small sample (4,397 combined rows) -- champion selection evidence is directional, "
        "not a large-sample statistical guarantee.",
        "customer_id is sheet-namespaced (Phase 2 decision) -- a real customer appearing in "
        "both CustomerViolation and TransactionNameViolation is treated as two separate "
        "histories until the client confirms the UIN spaces are the same.",
        "Character/token representation (E2-E5) uses a 50-component SVD reduction of the "
        "TF-IDF space for tractability -- champion evidence is about this reduced "
        "representation, not the full TF-IDF space directly.",
    ],
    "next_gate": "Phase 6 -- Transaction/Rule model experiments (Isolation Forest + Autoencoder "
        "benchmarks on the Phase 4 transaction representation). Requires human approval.",
}

out_path = REPO_ROOT / "evaluation" / "phase5_entity_experiments_report.json"
with open(out_path, "w") as f:
    json.dump(phase5_report, f, indent=2, default=str)
print(f"Report written to {out_path}")

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase5_entity_experiments_report.json


## 11. Test suite

In [14]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "tests/", "-q"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 5 is considered done" 

........................................................................ [ 62%]
...........................................                              [100%]
=============================== warnings summary ===============================
tests/test_anomaly_models.py::test_autoencoder_fit_score_shapes
  /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
    warnings.warn("Can't initialize NVML")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
115 passed, 1 warning in 11.96s



## Phase 5 -- Result

**Status: PASS**

- Trained and compared 5 candidates (Isolation Forest on tabular features, Isolation Forest / Autoencoder / LOF / One-Class SVM on the character-token name representation) under 2 validation scenarios (unseen customers, repeat-customer time-forward) -- 10 experiments total.
- Isolation Forest was **not** assumed the winner going in; champion selected via a documented rubric (non-degenerate distribution -> bootstrap ranking stability -> repeat-customer consistency -> cost), never accuracy, never Status/UPS/Released.
- The one cell that touches historical Status is explicitly labeled exploratory-only and structurally cannot feed selection (joined back in after scoring, from a column deliberately excluded from the feature-building dataset).
- Manual inspection (top-N, segment novelty, temporal stability) uses only non-identifying columns -- no names, DOB, or UIN anywhere in this notebook's output, per the PII lesson from Phase 2.
- Champion model + full manifest persisted (gitignored, local-only).
- 115+/115+ tests passing.

**Next gate:** Phase 6 -- Transaction/Rule model experiments on the Phase 4 behavioural representation. Awaiting human approval to proceed.